In [ ]:
"""
Please ntoe that, in this application, we consider an simple RAG architecture, by directly feeding text to embedding model, without dividing into chuncks, and we do not consider to feed chat history back into the prompt. In later developemnts we will consider advanced RAG architecutre, including those features. 
We use OPENAI embedding model and a OPenAI LLM model to deveopl this RAG app
to store ata in vector format we user chromaDB vector database type.
"""


In [3]:
#install necessary packages for openAI emedidng/LLM models and chroma vectordatabase
!pip install langchain -qU
!pip install langchain-openai -qU 
!pip install langchain-chroma -qU


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
#import librares
import os

In [6]:
#call the API key as an environment variable
#to manage API key as a local enviornment variable we need OS library, and to load the env variables from .env file we need to install python-dotenv package
%pip install python-dotenv

#load openai key from .env file, first import the library to load env variables
from dotenv import load_dotenv
from pathlib import Path

env_path=Path.cwd() / 'Chatbot' / '.env'#give the .env file path
print("Path to .env file:", env_path)
print("File exists:", env_path.exists())
# Load environment variables from .env file
load_dotenv(env_path,override=True)


#OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
api_key = os.getenv("OPENAI_API_KEY")
print("Loaded:", api_key is not None)

Note: you may need to restart the kernel to use updated packages.
Path to .env file: c:\Users\shara\OneDrive\Documents\Coding Stuff\Generative AI\Chatbot\.env
File exists: True
Loaded: True



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
#initialize the chatOPENAI model
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.9, openai_api_key=api_key)

In [24]:
#initialize the embedding model, here also we use a embedding model from openAI
from langchain_openai import OpenAIEmbeddings
embedding_model=OpenAIEmbeddings(model="text-embedding-3-small")


In [18]:
#create embedding for documents
# here instead of loading from html, or other data soruces, here we directly load data into document variables in the langchain. Later we would discuss advance data loading methods form other soruces.
from langchain_core.documents import Document

#define a list of documents with conent and data
documents=[
    Document(page_content='The Middle Eastern crisis is an ongoing series of interrelated wars, conflicts, and heightened instability in the Middle East as a result of the Gaza war and genocide. It has primarily consisted of conflicts between Israel and Iran-backed militias that form the "Axis of Resistance", including Hamas in the Gaza Strip,[i] Hezbollah in Lebanon, and the Houthis in Yemen; Iran itself has also been involved. Allies of Israel, including the United States, the United Kingdom, and France, have also intervened militarily in various theaters. The crisis has involved nearly all Middle Eastern countries, significantly affecting the region as a whole.',
             metadata={"source":"middleEastern crisis news"},
            ),
    Document(page_content='Sri Lanka hit the winning groove bleeding their England counterparts by 52 runs anchored by Limansa Tillekaratne who struck 51 off 85 balls with 2 fours and a six in a score of 170 in 42.4 overs despite a telling 4 for 9 grab by Trudy Johnson in a Womens U19 Tri Nation fixture at the Ian Healy Oval Grounds at Brisbane, Australia.',
             metadata={"source":"cricket news"},
            ),
    Document(page_content='BTS released their fifth studio album, ARIRANG, on March 20, 2026, featuring the lead single "Swim" an alternative synth-pop anthem about resilience. The album includes tracks like "2.0" and a fan-dedicated song comeover produced by "SUGA"',
             metadata={"source":"Music news"},
            ),
    Document(page_content='AI agents hold the promise of automatically moving data between systems and triggering decisions, but in some cases, they can act without a clear record of what, when, and why they undertook their tasks.',
             metadata={"source":"AI news"},
            ),     
]

In [25]:
# save these docuemnts in the vector database using embedding model
from langchain_chroma import Chroma
vectorstore=Chroma.from_documents(documents, embedding=embedding_model)


In [27]:
#lets perform a similarity search with the provided documentd and a text input which we give

results=vectorstore.similarity_search("middle east") #result will assign documents based on their match to the given text "middle east"

for result in results:
    print('-----------------')
    print(result.page_content)
    print(result.metadata)


results2=vectorstore.similarity_search("Korea")
for result in results2:
    print('-----------------')
    print(result.page_content)
    print(result.metadata)

-----------------
The Middle Eastern crisis is an ongoing series of interrelated wars, conflicts, and heightened instability in the Middle East as a result of the Gaza war and genocide. It has primarily consisted of conflicts between Israel and Iran-backed militias that form the "Axis of Resistance", including Hamas in the Gaza Strip,[i] Hezbollah in Lebanon, and the Houthis in Yemen; Iran itself has also been involved. Allies of Israel, including the United States, the United Kingdom, and France, have also intervened militarily in various theaters. The crisis has involved nearly all Middle Eastern countries, significantly affecting the region as a whole.
{'source': 'middleEastern crisis news'}
-----------------
AI agents hold the promise of automatically moving data between systems and triggering decisions, but in some cases, they can act without a clear record of what, when, and why they undertook their tasks.
{'source': 'AI news'}
-----------------
BTS released their fifth studio al

In [ ]:
# we also can give the embedding and check the similarity
query_embedding=embedding_model.embed_query("korea")
len(query_embedding) # length of selected openAI embedding query system is 1536



1536

In [31]:
query_embedding[:10]# first ten values of the embedding

[-0.005428314208984375,
 -0.01181793212890625,
 -0.005275726318359375,
 0.01078033447265625,
 -0.036865234375,
 0.0009264945983886719,
 -0.052459716796875,
 0.0751953125,
 -0.03814697265625,
 -0.00928497314453125]

In [33]:
# now we can use that embedding to find the similarity
results3=vectorstore.similarity_search_by_vector(query_embedding)
for result in results3:
    print('-----------------')
    print(result.page_content)
    print(result.metadata)

-----------------
BTS released their fifth studio album, ARIRANG, on March 20, 2026, featuring the lead single "Swim" an alternative synth-pop anthem about resilience. The album includes tracks like "2.0" and a fan-dedicated song comeover produced by "SUGA"
{'source': 'Music news'}
-----------------
Sri Lanka hit the winning groove bleeding their England counterparts by 52 runs anchored by Limansa Tillekaratne who struck 51 off 85 balls with 2 fours and a six in a score of 170 in 42.4 overs despite a telling 4 for 9 grab by Trudy Johnson in a Womens U19 Tri Nation fixture at the Ian Healy Oval Grounds at Brisbane, Australia.
{'source': 'cricket news'}
-----------------
The Middle Eastern crisis is an ongoing series of interrelated wars, conflicts, and heightened instability in the Middle East as a result of the Gaza war and genocide. It has primarily consisted of conflicts between Israel and Iran-backed militias that form the "Axis of Resistance", including Hamas in the Gaza Strip,[i] 

In [40]:
#Now let us create a retriver to retrive data from the vector database. for that we create a retirever object
retriever=vectorstore.as_retriever(search_type="similarity", search_kwargs={"k":1})#k menas number of results which should retriver)
#perform a batch retrieval using the retriver

batch_results=retriever.batch(["korea","Lemansa"])

for result in batch_results:
    print("--------------------------")
    for doc in result:
        print(doc.page_content)
        print(doc.metadata)

--------------------------
BTS released their fifth studio album, ARIRANG, on March 20, 2026, featuring the lead single "Swim" an alternative synth-pop anthem about resilience. The album includes tracks like "2.0" and a fan-dedicated song comeover produced by "SUGA"
{'source': 'Music news'}
--------------------------
Sri Lanka hit the winning groove bleeding their England counterparts by 52 runs anchored by Limansa Tillekaratne who struck 51 off 85 balls with 2 fours and a six in a score of 170 in 42.4 overs despite a telling 4 for 9 grab by Trudy Johnson in a Womens U19 Tri Nation fixture at the Ian Healy Oval Grounds at Brisbane, Australia.
{'source': 'cricket news'}


In [ ]:
#Now we can create a prompt template as per our requirements
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

prompt = ChatPromptTemplate.from_messages([("human", "You are an intelligent chatbot. Answer the following question, using provided context only {question} Context:{context}")])#human message to set the context for the chatbot with a variable {question} to take the user input

In [44]:
#chain retirever and prompt template with LLM
# here we add the retriver, then prompt and then LLM. 
chain={"context":retriever,"question":RunnablePassthrough()}|prompt|llm # here we used the runnable pass through menas the text which user input text to the system

In [46]:
#check on a text input formthe user
response=chain.invoke("current war status in middle east")
print(response.content)

The current war status in the Middle East is marked by ongoing conflicts between Israel and Iran-backed militias such as Hamas, Hezbollah, and the Houthis. The crisis has drawn in allies of Israel, including the United States, the United Kingdom, and France, leading to heightened instability in the region. The conflict has significantly affected nearly all Middle Eastern countries, creating a complex and volatile situation.
